# 🐝 Lab 5.1 — LoRA & Friends: Fine-Tuning a Small LLM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varoonsahgal/advancedml/blob/main/notebooks/lab13_lora_finetune.ipynb)

⏱ **Time:** ~2 hours  |  💻 **Runs on:** CPU (a GPU just makes it faster; the QLoRA cell is ⚡ GPU-only)

### 🎯 Learning objectives
- Explain **why low-rank updates work** for fine-tuning and do the parameter-count math for LoRA.
- Implement a **LoRA linear layer from scratch** — initialization, the $\alpha/r$ scaling, and **merge / unmerge** — and prove the merged weights give identical outputs.
- Fine-tune **SmolLM2-135M-Instruct** with your own LoRA on a small instruction dataset, using **chat templates** and **loss masking** of prompt tokens.
- Reproduce the same adapter with the Hugging Face **PEFT** library and check it matches your implementation exactly.
- Place LoRA in the 2026 post-training landscape: **QLoRA**, **DoRA**, and **SFT vs. preference tuning**.

### 🧰 Prerequisites
PyTorch `nn.Module`s and training loops, matrix rank / SVD at the "I've seen it" level, and the basics of causal language modeling (next-token cross-entropy).

### 🌍 Why this matters in 2026
Almost nobody fine-tunes all the weights of a large model any more. Adapter methods — LoRA and its descendants — let you customize a model on a single GPU (or a laptop CPU, as you will see), ship the customization as a **few-megabyte file**, and hot-swap dozens of adapters on one served base model. Open-weight models, on-device assistants and most enterprise "fine-tune on our data" offerings are built on this trick. Buzz the Backprop Bee 🐝 will teach a tiny model a new job — turning messy beekeeping notes into clean JSON — by training less than half a percent of its weights.

## 🗺️ Roadmap
| Part | What you'll do | ⏱ min |
|---|---|---|
| 0 | Setup (install `peft` if needed) | 5 |
| 1 | Low-rank intuition and LoRA parameter math | 15 |
| 2 | Build a LoRA layer from scratch, with merge/unmerge | 20 |
| 3 | Toy experiment: does rank matter? (with an animation of LoRA learning) | 15 |
| 4 | SmolLM2, chat templates, loss masking, baseline eval | 20 |
| 5 | Fine-tune with *your* LoRA, then merge for deployment | 20 |
| 6 | Reproduce it with 🤗 PEFT and check the outputs match exactly | 10 |
| 7 | QLoRA, DoRA (build one), SFT vs preference tuning | 15 |

## 0 · Setup
`peft` is installed only if missing. The model we use (`HuggingFaceTB/SmolLM2-135M-Instruct`) is a ~270 MB download.

In [ ]:
import importlib.util
if importlib.util.find_spec("peft") is None:
    %pip install -q peft

In [ ]:
import os, sys, math, random, time, json, re
import numpy as np
import torch
import matplotlib.pyplot as plt

FAST = os.environ.get("LAB_FAST", "0") == "1"   # smoke-test mode (normal users: leave unset)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed: int = 42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)

HIVE = ["#F5B700", "#1F1B16", "#FF6B5B", "#1FA39A", "#3D4DB7", "#9B5DE5"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=HIVE), "figure.facecolor": "#FFFBEA",
                     "axes.facecolor": "#FFFDF5", "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 11})
print(f"🐝 torch {torch.__version__} | device={DEVICE} | FAST={FAST}")

import copy
import torch.nn as nn
import torch.nn.functional as F
torch.set_num_threads(int(os.environ.get("OMP_NUM_THREADS", min(8, os.cpu_count() or 1))))

import warnings
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()
try:
    from huggingface_hub.utils import logging as hub_logging
    hub_logging.set_verbosity_error()
except Exception:
    pass
warnings.filterwarnings("ignore", category=UserWarning, module="transformers")

MODEL_ID   = "HuggingFaceTB/SmolLM2-135M-Instruct"
TRAIN_STEPS = 8 if FAST else 40
N_EVAL      = 4 if FAST else 12

## Part 1 · Why low-rank updates? 🧮

Fine-tuning changes a pretrained weight matrix $W_0 \in \mathbb{R}^{d_\text{out}\times d_\text{in}}$ into $W_0 + \Delta W$. Full fine-tuning learns all $d_\text{out} d_\text{in}$ entries of $\Delta W$ (and stores optimizer state for each — Adam keeps two more copies!).

**Key empirical observation** (Aghajanyan et al., 2020; Hu et al., 2021): the updates needed to adapt a pretrained model to a downstream task have **low intrinsic dimension**. So **LoRA** parameterizes the update as a product of two thin matrices:

$$\Delta W = \frac{\alpha}{r}\, B A, \qquad B \in \mathbb{R}^{d_\text{out}\times r},\; A \in \mathbb{R}^{r\times d_\text{in}},\; r \ll \min(d_\text{in}, d_\text{out}).$$

The forward pass becomes $h = W_0 x + \frac{\alpha}{r} B A x$, with $W_0$ **frozen**.

| | trainable params per matrix |
|---|---|
| full fine-tune | $d_\text{out}\, d_\text{in}$ |
| LoRA rank $r$ | $r\,(d_\text{in} + d_\text{out})$ |

For a $4096\times4096$ attention projection, full = 16.8M parameters; LoRA with $r=8$ = 65.5K — a **256× reduction**.

### 🧪 Your Turn #1 — parameter counting
Write `lora_param_count(d_in, d_out, r)` and `compression(d_in, d_out, r)` (= full / LoRA).

In [ ]:
def lora_param_count(d_in: int, d_out: int, r: int) -> int:
    # TODO: number of trainable parameters in A (r x d_in) and B (d_out x r)
    raise NotImplementedError

def compression(d_in: int, d_out: int, r: int) -> float:
    # TODO: full-fine-tune parameter count divided by LoRA parameter count
    raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
def lora_param_count(d_in: int, d_out: int, r: int) -> int:
    return r * d_in + d_out * r

def compression(d_in: int, d_out: int, r: int) -> float:
    return (d_in * d_out) / lora_param_count(d_in, d_out, r)

In [ ]:
# ✔️ Checkpoint
assert lora_param_count(4096, 4096, 8) == 65_536
assert abs(compression(4096, 4096, 8) - 256.0) < 1e-9
assert lora_param_count(576, 192, 4) == 4 * (576 + 192)
print("✔️ parameter math checks out")

ranks = [1, 2, 4, 8, 16, 32, 64, 128]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for d in [576, 4096, 8192]:
    ax[0].plot(ranks, [lora_param_count(d, d, r) for r in ranks], "o-", label=f"d={d}")
    ax[1].plot(ranks, [compression(d, d, r) for r in ranks], "o-", label=f"d={d}")
ax[0].set_xscale("log", base=2); ax[0].set_yscale("log"); ax[0].set_title("LoRA trainable params (one d×d matrix)"); ax[0].set_xlabel("rank r")
ax[1].set_xscale("log", base=2); ax[1].set_yscale("log"); ax[1].set_title("compression vs full fine-tune"); ax[1].set_xlabel("rank r")
ax[0].legend(); ax[1].legend(); plt.tight_layout(); plt.show()

> **🍯 Key takeaway:** LoRA trains $r(d_\text{in}+d_\text{out})$ numbers per matrix instead of $d_\text{in}d_\text{out}$: hundreds of times fewer, because fine-tuning updates live in a low-dimensional subspace.

## Part 2 · A LoRA layer from scratch 🛠️

Design decisions (matching the LoRA paper and the PEFT library):

- **Init:** $A$ random (Kaiming-uniform), $B = 0$. Then $BA = 0$ at the start, so the adapted model is **exactly** the pretrained model at step 0 — training starts from a known-good point. (Why not both random? You'd inject noise. Why not both zero? Then the gradients of both $A$ and $B$ are zero and nothing ever learns!)
- **Scaling** $\alpha / r$: lets you change $r$ without re-tuning the learning rate much. (rsLoRA, Kalajdzievski 2023, argues for $\alpha/\sqrt{r}$ at high ranks.)
- **Dropout** on the adapter input (optional regularizer).
- **Merge:** for deployment, fold the update into the base weight: $W \leftarrow W_0 + \frac{\alpha}{r}BA$. Inference then costs *exactly* what the base model costs — zero added latency. **Unmerge** subtracts it back (to swap adapters).

### 🧪 Your Turn #2 — implement `forward`, `merge`, `unmerge`

In [ ]:
class LoRALinear(nn.Module):
    """Wraps a frozen nn.Linear with a trainable low-rank update (alpha/r) * B @ A."""
    def __init__(self, base: nn.Linear, r: int = 8, alpha: float = 16.0, dropout: float = 0.0):
        super().__init__()
        self.base, self.r, self.alpha = base, r, alpha
        self.scaling = alpha / r
        for p in self.base.parameters():
            p.requires_grad_(False)                        # W0 (and bias) frozen
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))  # same init as nn.Linear / PEFT
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.merged = False

    def delta_weight(self):
        return (self.B @ self.A) * self.scaling            # (d_out, d_in)

    def forward(self, x):
        # TODO: if merged, the base layer already contains the update -> just return base(x)
        # TODO: otherwise return base(x) + scaling * (dropout(x) @ A^T @ B^T)
        raise NotImplementedError

    @torch.no_grad()
    def merge(self):
        # TODO: add delta_weight() into self.base.weight (once!) and set self.merged = True
        raise NotImplementedError

    @torch.no_grad()
    def unmerge(self):
        # TODO: subtract delta_weight() from self.base.weight and set self.merged = False
        raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
class LoRALinear(nn.Module):
    """Wraps a frozen nn.Linear with a trainable low-rank update (alpha/r) * B @ A."""
    def __init__(self, base: nn.Linear, r: int = 8, alpha: float = 16.0, dropout: float = 0.0):
        super().__init__()
        self.base, self.r, self.alpha = base, r, alpha
        self.scaling = alpha / r
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.A = nn.Parameter(torch.empty(r, base.in_features, dtype=base.weight.dtype))
        self.B = nn.Parameter(torch.zeros(base.out_features, r, dtype=base.weight.dtype))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.merged = False

    def delta_weight(self):
        return (self.B @ self.A) * self.scaling

    def forward(self, x):
        out = self.base(x)
        if self.merged:
            return out
        return out + (self.dropout(x) @ self.A.T @ self.B.T) * self.scaling

    @torch.no_grad()
    def merge(self):
        if not self.merged:
            self.base.weight += self.delta_weight()
            self.merged = True

    @torch.no_grad()
    def unmerge(self):
        if self.merged:
            self.base.weight -= self.delta_weight()
            self.merged = False

    def extra_repr(self):
        return f"r={self.r}, alpha={self.alpha}, merged={self.merged}"

In [ ]:
# ✔️ Checkpoint
seed_everything(0)
base = nn.Linear(64, 32)
W0 = base.weight.detach().clone()
lora = LoRALinear(copy.deepcopy(base), r=4, alpha=8)
x = torch.randn(10, 64)
assert torch.allclose(lora(x), base(x)), "B=0 at init -> identical to the base layer"
assert sum(p.numel() for p in lora.parameters() if p.requires_grad) == lora_param_count(64, 32, 4)

with torch.no_grad():
    lora.B.normal_()                                   # pretend we trained it
y_unmerged = lora(x)
lora.merge()
y_merged = lora(x)
assert torch.allclose(y_unmerged, y_merged, atol=1e-5), "merged weights must give identical outputs"
lora.merge()                                           # merging twice must be a no-op
assert torch.allclose(lora(x), y_merged, atol=1e-5)
lora.unmerge()
assert torch.allclose(lora.base.weight, W0, atol=1e-6), "unmerge restores W0"
print("✔️ LoRA layer: identity at init, merge == unmerged, unmerge restores W0  🐝")

> **🍯 Key takeaway:** With $B=0$ at init the adapted model *is* the pretrained model at step 0, and merging $W_0 + \frac{\alpha}{r}BA$ makes the adapter free at inference time.

## Part 3 · Does rank matter? A toy experiment

Let's make a world where we *know* the answer. A "pretrained" layer $W_0$ must be adapted to a target $W^\star = W_0 + \Delta^\star$ where $\Delta^\star$ has **rank 4** (plus a little full-rank noise). We'll

1. fit the update with **full fine-tuning** and look at the **singular values** of the learned $\Delta W$, then
2. fit it with LoRA at ranks 1, 2, 4, 8.

### 🔮 Predict-then-run
How many singular values of the fully fine-tuned $\Delta W$ will be "big"? Which LoRA ranks will reach the noise floor?

In [ ]:
seed_everything(1)
d_in, d_out, n = 64, 64, 2048
W0 = torch.randn(d_out, d_in) / math.sqrt(d_in)
U, V = torch.randn(d_out, 4), torch.randn(4, d_in)
delta_star = (U @ V) / math.sqrt(d_in) + 0.01 * torch.randn(d_out, d_in)
X = torch.randn(n, d_in)
Y = X @ (W0 + delta_star).T

def fit(layer, steps=400 if FAST else 1500, lr=1e-2):
    opt = torch.optim.Adam([p for p in layer.parameters() if p.requires_grad], lr=lr)
    losses = []
    for _ in range(steps):
        loss = F.mse_loss(layer(X), Y)
        opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
    return losses

# 1) full fine-tuning
full = nn.Linear(d_in, d_out, bias=False); full.weight.data = W0.clone()
full_losses = fit(full)
S = torch.linalg.svdvals(full.weight.detach() - W0)

# 2) LoRA at several ranks
curves = {}
for r in [1, 2, 4, 8]:
    seed_everything(2)
    b = nn.Linear(d_in, d_out, bias=False); b.weight.data = W0.clone()
    curves[r] = fit(LoRALinear(b, r=r, alpha=r))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].bar(range(1, 21), S[:20].numpy(), color=HIVE[0]); ax[0].set_title("singular values of full-FT ΔW"); ax[0].set_xlabel("index")
for r, c in curves.items():
    ax[1].plot(c, label=f"LoRA r={r}")
ax[1].plot(full_losses, "k--", label="full fine-tune")
ax[1].set_yscale("log"); ax[1].set_title("training MSE"); ax[1].set_xlabel("step"); ax[1].legend()
plt.tight_layout(); plt.show()
print({r: f"{c[-1]:.2e}" for r, c in curves.items()}, "full:", f"{full_losses[-1]:.2e}")

The full fine-tune "discovers" that only ~4 directions matter — the rest of its singular values are noise. LoRA with $r \ge 4$ captures the whole structured part of the update with a fraction of the parameters and stalls at the **noise floor** (≈ the energy of the small full-rank noise, which only full fine-tuning can fit); $r<4$ cannot represent the update at all. Real fine-tuning updates aren't exactly low-rank, but their spectrum decays fast enough that $r\in[4, 64]$ usually suffices. (Biderman et al., 2024, show the flip side: LoRA "learns less and forgets less" than full fine-tuning on hard domains like code and math.)

### 🎬 Watch LoRA learn
Let's *see* it. Below, the left panel is the true update $\Delta^\star$ (rank 4 plus a little noise). The middle and right panels show the learned $\frac{\alpha}{r}BA$ for $r=1$ and $r=4$ as training proceeds. Rank 1 can only paint a single "outer-product stripe pattern"; rank 4 reproduces the whole structure.

In [ ]:
from matplotlib import animation
from IPython.display import HTML, display

def lora_snapshots(r, steps=240, every=12):
    seed_everything(2)
    b = nn.Linear(d_in, d_out, bias=False); b.weight.data = W0.clone()
    layer = LoRALinear(b, r=r, alpha=r)
    opt = torch.optim.Adam([layer.A, layer.B], lr=1e-2)
    snaps = []
    for s in range(steps + 1):
        if s % every == 0:
            snaps.append((s, layer.delta_weight().detach().clone(), F.mse_loss(layer(X), Y).item()))
        loss = F.mse_loss(layer(X), Y)
        opt.zero_grad(); loss.backward(); opt.step()
    return snaps

S1, S4 = lora_snapshots(1), lora_snapshots(4)
vmax = 0.6 * delta_star.abs().max().item()     # a bit of saturation makes the structure easier to see
fig, axes = plt.subplots(1, 3, figsize=(11, 3.9))
axes[0].imshow(delta_star, cmap="RdBu_r", vmin=-vmax, vmax=vmax); axes[0].set_title("true update Δ*  (rank 4)")
im1 = axes[1].imshow(S1[0][1], cmap="RdBu_r", vmin=-vmax, vmax=vmax)
im4 = axes[2].imshow(S4[0][1], cmap="RdBu_r", vmin=-vmax, vmax=vmax)
for a in axes: a.set_xticks([]); a.set_yticks([])
def update(i):
    (s, d1, l1), (_, d4, l4) = S1[i], S4[i]
    im1.set_data(d1); im4.set_data(d4)
    axes[1].set_title(f"LoRA r=1   step {s}   MSE {l1:.3f}")
    axes[2].set_title(f"LoRA r=4   step {s}   MSE {l4:.3f}")
    return im1, im4
anim = animation.FuncAnimation(fig, update, frames=len(S1), interval=180, blit=False)
display(HTML(anim.to_jshtml()))
update(len(S1) - 1); fig.suptitle("Learned update (α/r)·BA after training", y=1.02); plt.tight_layout(); plt.show()

> **🍯 Key takeaway:** When the true update has rank $k$, LoRA with $r \ge k$ captures it and $r<k$ cannot. Real fine-tuning updates have fast-decaying spectra, so small ranks usually suffice.

## Part 4 · A real model, a real task: SmolLM2 + chat templates 🍯

**Task — "HiveLog" structured extraction.** Beekeepers scribble notes like

> *Hive #42: 7 brood frames, bees were grumpy. Saw the queen!*

and we want exactly

> `{"hive_id": 42, "queen_seen": true, "brood_frames": 7, "mood": "grumpy"}`

We generate a few hundred synthetic examples inline (varied templates, so the model can't just memorize one sentence shape).

**Chat templates.** Instruct models are trained on conversations rendered with special tokens (SmolLM2 uses ChatML: `<|im_start|>role ... <|im_end|>`). You **must** fine-tune with the *same* template the model was trained with — `tokenizer.apply_chat_template` does it for you.

**Loss masking.** We only want to learn to produce the *assistant* reply, not to predict the system prompt or the user's note. So labels for prompt tokens are set to `-100` (PyTorch's `ignore_index`):

$$\mathcal{L} = -\frac{1}{|\mathcal{R}|}\sum_{t\in\mathcal{R}} \log p_\theta(x_t \mid x_{<t}), \qquad \mathcal{R} = \text{response tokens}.$$

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID).float().to(DEVICE).eval()   # fp32 for CPU training
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
n_params = sum(p.numel() for p in model.parameters())
print(f"{MODEL_ID}: {n_params/1e6:.1f}M params, {model.config.num_hidden_layers} layers, hidden={model.config.hidden_size}")
print(model.model.layers[0])

In [ ]:
MOODS = ["calm", "buzzy", "grumpy", "sleepy"]
TEMPLATES = [
    "Inspected hive {h} today. {q} {f} frames of brood. The colony seemed {m}.",
    "Hive #{h}: {f} brood frames, bees were {m}. {q}",
    "Quick note - colony {h} was {m}; counted {f} frames with brood. {q}",
    "{q} Hive {h} has {f} frames of brood and the bees are {m}.",
    "Visited box {h}. Mood: {m}. Brood on {f} frames. {q}",
]
def make_example(rng):
    h, queen, f, m = rng.randint(1, 99), rng.random() < 0.5, rng.randint(0, 10), rng.choice(MOODS)
    q = rng.choice(["Queen spotted.", "Saw the queen!", "The queen was there."]) if queen else \
        rng.choice(["No queen seen.", "Couldn't find the queen.", "Queen not spotted."])
    note = rng.choice(TEMPLATES).format(h=h, q=q, f=f, m=m)
    target = json.dumps({"hive_id": h, "queen_seen": queen, "brood_frames": f, "mood": m})
    return note, target

SYSTEM = ("You extract beekeeping inspection notes into JSON with keys hive_id (int), queen_seen (bool), "
          "brood_frames (int), mood (str). Reply with JSON only.")
def to_messages(note, target=None):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": note}]
    if target is not None:
        msgs.append({"role": "assistant", "content": target})
    return msgs

rng = random.Random(0)
train_data = [make_example(rng) for _ in range(300)]
rng = random.Random(1000)
test_data = [make_example(rng) for _ in range(50)]
for note, target in train_data[:3]:
    print(f"📝 {note}\n   → {target}")
print()
print(tok.apply_chat_template(to_messages(*train_data[0]), tokenize=False))

### 🧪 Your Turn #3 — tokenization with loss masking
Return `(input_ids, labels)` for one example where `labels` equals `input_ids` on the assistant reply and `-100` on every prompt token.

*Hints:* render the prompt with `add_generation_prompt=True` (this appends the `<|im_start|>assistant` header) and the full conversation without it. The prompt ids should be a **prefix** of the full ids — check it!

In [ ]:
def _ids(x):
    """apply_chat_template(tokenize=True) returns a list in some versions and a dict-like in others."""
    return list(x["input_ids"]) if hasattr(x, "keys") else list(x)

def build_example(note, target):
    # TODO 1: prompt_ids = _ids(tok.apply_chat_template(to_messages(note), add_generation_prompt=True, tokenize=True))
    # TODO 2: full_ids   = _ids(tok.apply_chat_template(to_messages(note, target), tokenize=True))
    # TODO 3: assert the prompt is a prefix of the full sequence
    # TODO 4: labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
def build_example(note, target):
    prompt_ids = _ids(tok.apply_chat_template(to_messages(note), add_generation_prompt=True, tokenize=True))
    full_ids = _ids(tok.apply_chat_template(to_messages(note, target), tokenize=True))
    assert full_ids[:len(prompt_ids)] == prompt_ids, "chat template prompt must be a prefix of the full conversation"
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    return full_ids, labels

In [ ]:
# ✔️ Checkpoint
ids, labels = build_example(*train_data[0])
assert len(ids) == len(labels)
learned = [t for t in labels if t != -100]
decoded = tok.decode(learned)
print("tokens:", len(ids), "| supervised:", len(learned), "| supervised text:", repr(decoded))
assert decoded.startswith(train_data[0][1]), "the supervised tokens should be exactly the assistant reply (+ end token)"
assert labels[0] == -100 and labels[-1] != -100
train_tok = [build_example(n, t) for n, t in train_data]
print("✔️ loss masking OK — the model will only learn to write the JSON")

### Baseline: how does the *untuned* model do?
We score each reply with two metrics:
- **strict JSON** — the whole reply parses with `json.loads` (what a downstream program needs), and
- **field accuracy** — fraction of the 4 fields correct, after leniently extracting the first `{...}` block.

In [ ]:
@torch.no_grad()
def generate_reply(model, note, max_new_tokens=48):
    enc = tok.apply_chat_template(to_messages(note), add_generation_prompt=True, return_tensors="pt", return_dict=True)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def score_reply(reply, target):
    gold = json.loads(target)
    try:
        json.loads(reply); strict = 1.0
    except Exception:
        strict = 0.0
    m = re.search(r"\{.*?\}", reply, flags=re.S)
    try:
        d = json.loads(m.group(0)) if m else {}
    except Exception:
        d = {}
    fields = np.mean([d.get(k) == v for k, v in gold.items()]) if isinstance(d, dict) else 0.0
    return strict, float(fields)

def evaluate(model, n=N_EVAL, show=2):
    model.eval(); rows = []
    for i, (note, target) in enumerate(test_data[:n]):
        reply = generate_reply(model, note)
        rows.append(score_reply(reply, target))
        if i < show:
            print(f"📝 {note}\n🤖 {reply!r}\n")
    strict, fields = np.mean(rows, axis=0)
    print(f"strict-JSON rate {strict:.0%} | field accuracy {fields:.0%}  (n={n})")
    return strict, fields

t0 = time.time()
base_scores = evaluate(model)
print(f"({time.time()-t0:.0f}s)")

> **🍯 Key takeaway:** Always fine-tune with the model's own chat template, and mask prompt tokens with `-100` so the loss teaches only the response.

## Part 5 · Fine-tune with *your* LoRA 🐝

We wrap the attention **query** and **value** projections of all 30 layers (the original LoRA paper's default) with `LoRALinear`, freeze everything else, and train with AdamW. Modern practice often targets *all* linear layers (q, k, v, o, gate, up, down) — try it in the Boss Level.

### 🔮 Predict-then-run
SmolLM2-135M has ~135M parameters. With $r=8$ on `q_proj` (576→576) and `v_proj` (576→192) in 30 layers, what fraction of the parameters will be trainable? Work it out with `lora_param_count` *before* running the cell.

In [ ]:
def inject_lora(model, target_names=("q_proj", "v_proj"), r=8, alpha=16, dropout=0.0):
    for p in model.parameters():
        p.requires_grad_(False)
    replaced = 0
    for parent in list(model.modules()):
        for name, child in list(parent.named_children()):
            if name in target_names and isinstance(child, nn.Linear):
                setattr(parent, name, LoRALinear(child, r=r, alpha=alpha, dropout=dropout))
                replaced += 1
    return replaced

def count_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

seed_everything(0)
lora_model = copy.deepcopy(model)
n_rep = inject_lora(lora_model, r=8, alpha=16)
trainable, total = count_params(lora_model)
cfg = model.config
expected = cfg.num_hidden_layers * (lora_param_count(cfg.hidden_size, cfg.hidden_size, 8) +
                                    lora_param_count(cfg.hidden_size, cfg.num_key_value_heads * (cfg.hidden_size // cfg.num_attention_heads), 8))
print(f"wrapped {n_rep} linear layers | trainable {trainable:,} / {total:,} = {trainable/total:.3%}")
assert trainable == expected, (trainable, expected)

In [ ]:
def collate(batch):
    T = max(len(ids) for ids, _ in batch)
    input_ids = torch.full((len(batch), T), tok.pad_token_id)
    labels = torch.full((len(batch), T), -100)
    attn = torch.zeros((len(batch), T), dtype=torch.long)
    for i, (ids, lab) in enumerate(batch):
        input_ids[i, :len(ids)] = torch.tensor(ids); labels[i, :len(lab)] = torch.tensor(lab); attn[i, :len(ids)] = 1
    return input_ids.to(DEVICE), attn.to(DEVICE), labels.to(DEVICE)

def train(model, steps=TRAIN_STEPS, lr=1e-3, batch_size=8, seed=0):
    rng = random.Random(seed)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    model.train(); losses = []; t0 = time.time()
    for step in range(steps):
        ids, attn, labels = collate(rng.sample(train_tok, batch_size))
        loss = model(input_ids=ids, attention_mask=attn, labels=labels).loss   # HF shifts labels internally
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
        if step % 5 == 0 or step == steps - 1:
            print(f"step {step:3d} | loss {loss.item():.4f} | {time.time()-t0:.0f}s")
    model.eval()
    return losses

lora_losses = train(lora_model)
plt.figure(figsize=(6, 3)); plt.plot(lora_losses, "o-"); plt.yscale("log")
plt.title("LoRA fine-tuning loss (response tokens only)"); plt.xlabel("step"); plt.show()
assert lora_losses[-1] < lora_losses[0], "loss should go down"

In [ ]:
lora_scores = evaluate(lora_model)
fig, ax = plt.subplots(figsize=(5.5, 3.2))
xs = np.arange(2)
ax.bar(xs - 0.18, base_scores, 0.36, label="base model", color=HIVE[2])
ax.bar(xs + 0.18, lora_scores, 0.36, label="+ our LoRA", color=HIVE[3])
ax.set_xticks(xs, ["strict JSON", "field accuracy"]); ax.set_ylim(0, 1.05); ax.legend(); ax.set_title("Before vs after LoRA")
plt.show()

### Merging for deployment
After training, fold every adapter into its base weight. The merged model must produce **identical logits** — and it no longer pays the extra matmuls.

In [ ]:
# ✔️ Checkpoint — merged model == adapter model
probe = tok(["Hive #7: 3 brood frames, bees were calm. Saw the queen!"], return_tensors="pt").to(DEVICE)
with torch.no_grad():
    logits_adapter = lora_model(**probe).logits
    for m in lora_model.modules():
        if isinstance(m, LoRALinear): m.merge()
    logits_merged = lora_model(**probe).logits
    for m in lora_model.modules():
        if isinstance(m, LoRALinear): m.unmerge()
    logits_unmerged = lora_model(**probe).logits
max_diff = (logits_adapter - logits_merged).abs().max().item()
print(f"max |logit difference| adapter vs merged: {max_diff:.2e}")
assert max_diff < 1e-3 and torch.allclose(logits_adapter, logits_unmerged, atol=1e-4)
print("✔️ merge/unmerge is exact (up to float32 round-off)")

> **🍯 Key takeaway:** Training 0.34% of SmolLM2's weights for a few dozen steps turned a chatty model that never produced valid JSON into a reliable extractor.

## Part 6 · The same thing with 🤗 PEFT

In practice you'll use the `peft` library: `LoraConfig` + `get_peft_model`. PEFT stores the adapter as `lora_A` ($r\times d_\text{in}$) and `lora_B` ($d_\text{out}\times r$) inside each wrapped layer — the same convention as ours. So we can **copy our trained $A, B$ into a PEFT model** and check that the outputs agree exactly. That's the strongest possible test of our implementation.

In [ ]:
from peft import LoraConfig, get_peft_model

peft_cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.0, target_modules=["q_proj", "v_proj"],
                      bias="none", task_type="CAUSAL_LM")
peft_model = get_peft_model(copy.deepcopy(model), peft_cfg)
peft_model.print_trainable_parameters()
peft_trainable, _ = count_params(peft_model)
assert peft_trainable == trainable, "PEFT and our implementation should train the same number of parameters"

# copy our trained adapters into PEFT's lora_A / lora_B
ours = {n: m for n, m in lora_model.named_modules() if isinstance(m, LoRALinear)}
copied = 0
with torch.no_grad():
    for name, mod in peft_model.named_modules():
        if hasattr(mod, "lora_A") and isinstance(getattr(mod, "lora_A"), nn.ModuleDict) and "default" in mod.lora_A:
            key = name.split("base_model.model.", 1)[-1]
            mod.lora_A["default"].weight.copy_(ours[key].A)
            mod.lora_B["default"].weight.copy_(ours[key].B)
            copied += 1
print("copied adapters:", copied)
with torch.no_grad():
    logits_peft = peft_model(**probe).logits
diff = (logits_peft - logits_adapter).abs().max().item()
print(f"max |logit difference| ours vs PEFT: {diff:.2e}")
assert copied == n_rep and diff < 1e-3
print("✔️ our LoRA == PEFT's LoRA")

In [ ]:
# Adapters are tiny on disk — that's why you can keep dozens per base model
import tempfile, pathlib
with tempfile.TemporaryDirectory() as d:
    peft_model.save_pretrained(d)
    files = {p.name: p.stat().st_size for p in pathlib.Path(d).iterdir()}
print({k: f"{v/1e6:.2f} MB" for k, v in files.items()})
print(f"vs the full fp32 model: {n_params*4/1e6:.0f} MB")

merged = peft_model.merge_and_unload()     # a plain transformers model with W0 + (alpha/r) BA baked in
print(type(merged).__name__, "— ready to serve with zero adapter overhead")

> **🍯 Key takeaway:** PEFT is the same math as your 40-line class (same `lora_A`/`lora_B` layout), and the whole customization ships as a ~2 MB file.

## Part 7 · LoRA's friends: QLoRA, DoRA, and where SFT stops

### QLoRA (Dettmers et al., 2023)
Store the frozen base weights in **4-bit NormalFloat (NF4)**, dequantize on the fly to bf16 for the matmul, and train LoRA adapters in higher precision on top. Plus **double quantization** (quantize the quantization constants) and **paged optimizers** to survive memory spikes. Result: fine-tuning a 65B model on one 48 GB GPU, matching 16-bit fine-tuning quality on their benchmarks. Gradients flow *through* the quantized weights into the adapters; the base never changes.

### DoRA (Liu et al., 2024)
Decompose each weight into a **magnitude** and a **direction**: $W = m \cdot \frac{V}{\lVert V\rVert}$. Train the magnitude vector $m$ directly and update the direction with LoRA:

$$W' = m \odot \frac{W_0 + \tfrac{\alpha}{r}BA}{\lVert W_0 + \tfrac{\alpha}{r}BA \rVert_{\text{row}}}, \qquad m_\text{init} = \lVert W_0\rVert_{\text{row}}.$$

(Norm over each output unit's row of the `(d_out, d_in)` PyTorch weight, as in PEFT's `use_dora=True`.) The authors find learning dynamics closer to full fine-tuning, often at the same rank.

### 🧪 Your Turn #4 — a DoRA layer
Implement `forward` for `DoRALinear`. At init ($B=0$) it must reproduce the base layer exactly.

In [ ]:
class DoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=8, alpha=16):
        super().__init__()
        self.base, self.scaling = base, alpha / r
        for p in base.parameters(): p.requires_grad_(False)
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.m = nn.Parameter(base.weight.detach().norm(dim=1).clone())        # (d_out,)

    def forward(self, x):
        # TODO 1: V = W0 + scaling * B @ A                       (d_out, d_in)
        # TODO 2: W = m[:, None] * V / ||V|| (norm of each row, keepdim)
        # TODO 3: return F.linear(x, W, self.base.bias)
        raise NotImplementedError

In [ ]:
# ✅ Solution (peek only after trying!)
class DoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=8, alpha=16):
        super().__init__()
        self.base, self.scaling = base, alpha / r
        for p in base.parameters(): p.requires_grad_(False)
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.m = nn.Parameter(base.weight.detach().norm(dim=1).clone())

    def forward(self, x):
        V = self.base.weight + self.scaling * (self.B @ self.A)
        W = self.m[:, None] * V / V.norm(dim=1, keepdim=True)
        return F.linear(x, W, self.base.bias)

In [ ]:
# ✔️ Checkpoint
seed_everything(3)
base = nn.Linear(32, 16)
dora = DoRALinear(copy.deepcopy(base), r=4, alpha=8)
x = torch.randn(5, 32)
assert torch.allclose(dora(x), base(x), atol=1e-5), "DoRA at init must equal the base layer"
with torch.no_grad():
    dora.B.normal_(); dora.m.mul_(2.0)
W_eff = dora.m[:, None] * F.normalize(base.weight + dora.scaling * dora.B @ dora.A, dim=1)
assert torch.allclose(dora(x), F.linear(x, W_eff, base.bias), atol=1e-5)
assert torch.allclose(W_eff.norm(dim=1), dora.m, atol=1e-5), "row norms of the effective weight == magnitude vector"
print("✔️ DoRA layer OK — magnitude and direction are decoupled")

### SFT vs. preference tuning
What you did today is **supervised fine-tuning (SFT)**: imitate target outputs token by token. It's perfect when you can *write down* the right answer (formats, extraction, style, a persona). It can't easily express "answer A is better than answer B" or "this is correct, however you phrase it". For that, modern pipelines add a second stage:

| stage | signal | examples |
|---|---|---|
| SFT | demonstrations | instruction tuning, today's JSON task |
| preference tuning | chosen ≻ rejected pairs | RLHF with PPO (Ouyang et al., 2022), **DPO** (Rafailov et al., 2023) |
| RL with verifiable rewards | a program checks the answer | GRPO for math/code reasoning → **Lab 5.2** |

All of these stack happily with LoRA: you can DPO- or GRPO-train just the adapters.

### ⚡ GPU recommended — QLoRA in five lines
This cell runs only when a CUDA GPU is present (bitsandbytes 4-bit kernels need one). On CPU it prints what it would do.

In [ ]:
if torch.cuda.is_available() and not FAST:
    if importlib.util.find_spec("bitsandbytes") is None:
        %pip install -q bitsandbytes
    from transformers import BitsAndBytesConfig
    from peft import prepare_model_for_kbit_training
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                             bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)
    qmodel = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
    qmodel = prepare_model_for_kbit_training(qmodel)
    qmodel = get_peft_model(qmodel, LoraConfig(r=16, lora_alpha=32, target_modules="all-linear", task_type="CAUSAL_LM"))
    qmodel.print_trainable_parameters()
    print(f"4-bit base footprint: {qmodel.get_memory_footprint()/1e6:.0f} MB")
else:
    print("⏭️  No GPU (or FAST mode): skipping QLoRA. On a T4 this loads the base in NF4 (~4x smaller) "
          "and attaches LoRA adapters to every linear layer.")

> **🍯 Key takeaway:** QLoRA saves memory (4-bit frozen base), DoRA changes the parameterization (magnitude × direction), and SFT teaches *what to say* while preference/RL stages teach *which answer is better*.

## 🏆 Boss Level
1. **Rank & targets sweep.** Train adapters with $r\in\{1, 4, 16\}$ on `("q_proj","v_proj")` vs. all linear layers `("q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")`. Plot field accuracy against trainable parameters. Where's the knee?
2. **Bee persona.** Build a second dataset where the assistant answers general questions *in character* as "Buzz" (bee puns, signs off with "🐝 Buzz buzz!"). Train a second adapter, then **hot-swap** adapters on the same base (`unmerge` one, load the other) and show both behaviors from one model. Measure persona adherence with a simple regex.
3. **Forgetting check.** Before and after fine-tuning, measure the model's loss on some generic text (e.g. a Wikipedia paragraph). How much general ability did the adapter cost? Does merging and then scaling the delta by 0.5 trade task accuracy for less forgetting?

## 🍯 Key Takeaways
- **Low-rank is enough:** fine-tuning updates have low intrinsic dimension, so $\Delta W = \frac{\alpha}{r}BA$ with $r \ll d$ captures most of what full fine-tuning learns.
- **Zero-init $B$** makes training start exactly at the pretrained model; random $A$ keeps gradients alive.
- **Merge for serving:** $W_0 + \frac{\alpha}{r}BA$ costs nothing extra at inference; **unmerge** to hot-swap adapters.
- **Chat template + loss masking** are not optional details: they decide what the model actually learns.
- **Tiny footprint:** 0.34% of SmolLM2's parameters and a ~2 MB adapter learned a new structured-output skill on a CPU.
- **Your code == PEFT:** same layout, identical logits; production libraries are not magic.
- **The family:** QLoRA (4-bit base), DoRA (magnitude/direction), rsLoRA/LoRA+ (better scaling and learning rates); SFT first, preference/RL tuning after.

## 🎤 Explain it to a friend
Your friend thinks "fine-tuning an AI" means retraining the whole brain on a supercomputer. In one paragraph with no equations, explain how you taught a 135-million-parameter model a new job by training only about one number in three hundred, why the result starts out *exactly* as smart as the original, and why the finished customization fits in a file smaller than a phone photo. (Hint: think of a clip-on lens for a camera, not a new camera.)

## 💼 Interview corner
**Q1. In LoRA, why is $B$ initialized to zero and $A$ randomly? What goes wrong if both are zero, or both random?**
<details><summary>Answer</summary>

With $B=0$, $BA=0$, so the adapted model equals the pretrained model at step 0: no random perturbation of a good solution. $A$ must be non-zero, otherwise $\partial L/\partial B = g\,(Ax)^\top = 0$ and $\partial L/\partial A = B^\top g\, x^\top = 0$: if both are zero, neither ever moves. If both are random, you inject a random update $BA$ at initialization that damages the pretrained behaviour before training even starts.
</details>

**Q2. Does LoRA slow down inference? How would you serve 50 customer-specific adapters on one base model?**
<details><summary>Answer</summary>

Merged, it adds zero latency because $W = W_0 + \frac{\alpha}{r}BA$ has the same shape as $W_0$. For many adapters you keep the base frozen in memory and either (a) merge/unmerge per request batch, or (b) keep adapters unmerged and compute $W_0x + \frac{\alpha}{r}B(Ax)$, batching requests that use different adapters (the idea behind multi-LoRA serving systems such as S-LoRA and Punica). The extra matmuls are cheap because $r$ is small, and each adapter is only megabytes.
</details>

**Q3. In QLoRA, what is quantized, what is trained, and how can gradients flow through 4-bit weights?**
<details><summary>Answer</summary>

The frozen base weights are stored in 4-bit NF4 (with double-quantized scaling constants); the LoRA adapters are trained in 16/32-bit. In the forward pass each 4-bit block is dequantized to bf16 for the matmul. Gradients flow *through* that dequantized weight to the activations and on to the adapters; the base weights themselves get no update, so they never need full-precision gradients or optimizer states. Paged optimizers absorb memory spikes.
</details>

## 🐝 Fun fact
The original LoRA paper reports that on GPT-3 175B, LoRA cut the number of trainable parameters by about **10,000×** and GPU memory during training by about **3×** compared with full fine-tuning, while matching or beating its quality on their benchmarks.

## 📚 Further reading
- Hu et al. (2021). *LoRA: Low-Rank Adaptation of Large Language Models.*
- Aghajanyan, Zettlemoyer & Gupta (2020). *Intrinsic Dimensionality Explains the Effectiveness of Language Model Fine-Tuning.*
- Dettmers et al. (2023). *QLoRA: Efficient Finetuning of Quantized LLMs.*
- Liu et al. (2024). *DoRA: Weight-Decomposed Low-Rank Adaptation.*
- Kalajdzievski (2023). *A Rank Stabilization Scaling Factor for Fine-Tuning with LoRA.* (rsLoRA)
- Hayou, Ghosh & Yu (2024). *LoRA+: Efficient Low Rank Adaptation of Large Models.*
- Biderman et al. (2024). *LoRA Learns Less and Forgets Less.*
- Houlsby et al. (2019). *Parameter-Efficient Transfer Learning for NLP.* (adapters)
- Allal et al. (2025). *SmolLM2: When Smol Goes Big — Data-Centric Training of a Small Language Model.*
- Rafailov et al. (2023). *Direct Preference Optimization: Your Language Model is Secretly a Reward Model.*
- Hugging Face PEFT documentation — LoRA, DoRA (`use_dora=True`) and QLoRA guides.